In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:

import tensorflow as tf

imagesize = (224, 224)
batchsize = 32
seed = 42

train_data_path = r"C:\Users\darsh\Documents\ML_Datasets\asl-alphabet\asl_alphabet_train\asl_alphabet_train"

test_data_path = r"C:\Users\darsh\Documents\ML_Datasets\asl-alphabet\asl_alphabet_test\asl_alphabet_test"

# Load training data: 80%
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_data_path,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=imagesize,
    batch_size=batchsize,
    label_mode="int"
)

# Load validation data: 20%
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_data_path,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=imagesize,
    batch_size=batchsize,
    label_mode="int"
)

#both train_ds and val_ds are from asl_alphabet_train dataset

# Get class names
class_names = train_ds.class_names

print("Classes:", class_names)
print("Number of classes:", len(class_names))

Found 87000 files belonging to 29 classes.
Using 69600 files for training.
Found 87000 files belonging to 29 classes.
Using 17400 files for validation.
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Number of classes: 29


In [3]:

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("First 10 labels:", labels[:10].numpy())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
First 10 labels: [25 23 20 16  4  2  2  8  0  1]


In [4]:
#preprocessing 

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Preprocess training images
train_ds = train_ds.map(
    lambda images, labels: (
        preprocess_input(images),
        labels
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Preprocess validation images
val_ds = val_ds.map(
    lambda images, labels: (
        preprocess_input(images),
        labels
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Improve input pipeline performance
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

val_ds = val_ds.prefetch(
    tf.data.AUTOTUNE
)

In [5]:

# Create a pretrained MobileNetV2 feature extractor

base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the pretrained CNN
base_model.trainable = False

# Define input
inputs = keras.Input(
    shape=(224, 224, 3)
)

# Extract spatial features
x = base_model(
    inputs,
    training=False
)

# Convert spatial feature maps into a vector
outputs = keras.layers.GlobalAveragePooling2D()(x)

# Build the feature extractor
feature_extractor = keras.Model(
    inputs,
    outputs,
    name="mobilenetv2_feature_extractor"
)

feature_extractor.summary()

Model: "mobilenetv2_feature_extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
#Creating the function which will extract and save features with MobileNetV2 architecture

import os
import numpy as np
import json

output_dir = r"C:\Users\darsh\Documents\signframe\cnn_model_asl-alphabet_dataset\extracted_features"

os.makedirs(output_dir, exist_ok=True)


def extract_and_save_features(
    dataset,
    feature_extractor,
    prefix,
    num_images
):

    feature_dim = feature_extractor.output_shape[-1]

    # Create memory-mapped NumPy files
    X = np.lib.format.open_memmap(
        os.path.join(
            output_dir,
            f"X_{prefix}.npy"
        ),
        mode="w+",
        dtype=np.float32,
        shape=(num_images, feature_dim)
    )

    y = np.lib.format.open_memmap(
        os.path.join(
            output_dir,
            f"y_{prefix}.npy"
        ),
        mode="w+",
        dtype=np.int32,
        shape=(num_images,)
    )

    index = 0

    for images, labels in dataset:

        # Extract features from the batch
        features = feature_extractor(
            images,
            training=False
        )

        features = features.numpy()
        labels = labels.numpy()

        batch_count = len(labels)

        # Save features and corresponding labels
        X[index:index + batch_count] = features
        y[index:index + batch_count] = labels

        index += batch_count

        if index % 3200 == 0:
            print(f"Processed {index} images")

    # Ensure every image was processed
    if index != num_images:
        raise ValueError(
            f"Expected {num_images} images, "
            f"but processed {index}"
        )

    X.flush()
    y.flush()

    print(f"\nFinished {prefix} extraction")
    print("Features:", X.shape)
    print("Labels:", y.shape)

    del X, y

In [7]:
#Calling func on train_ds and val_ds to compute and store features

extract_and_save_features(
    train_ds,
    feature_extractor,
    prefix="train",
    num_images=69600
)

extract_and_save_features(
    val_ds,
    feature_extractor,
    prefix="val",
    num_images=17400
)

Processed 3200 images
Processed 6400 images
Processed 9600 images
Processed 12800 images
Processed 16000 images
Processed 19200 images
Processed 22400 images
Processed 25600 images
Processed 28800 images
Processed 32000 images
Processed 35200 images
Processed 38400 images
Processed 41600 images
Processed 44800 images
Processed 48000 images
Processed 51200 images
Processed 54400 images
Processed 67200 images

Finished train extraction
Features: (69600, 1280)
Labels: (69600,)
Processed 3200 images
Processed 6400 images
Processed 9600 images
Processed 12800 images
Processed 16000 images

Finished val extraction
Features: (17400, 1280)
Labels: (17400,)


In [8]:

with open(
    os.path.join(
        output_dir,
        "class_names.json"
    ),
    "w"
) as f:

    json.dump(class_names, f)

print("Class mapping saved.")

Class mapping saved.


In [9]:
#loading precomputed CNN features

X_train = np.load(
    os.path.join(output_dir, "X_train.npy"),
    mmap_mode="r"
)

y_train = np.load(
    os.path.join(output_dir, "y_train.npy"),
    mmap_mode="r"
)

X_val = np.load(
    os.path.join(output_dir, "X_val.npy"),
    mmap_mode="r"
)

y_val = np.load(
    os.path.join(output_dir, "y_val.npy"),
    mmap_mode="r"
)

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

print("Validation features:", X_val.shape)
print("Validation labels:", y_val.shape)

Training features: (69600, 1280)
Training labels: (69600,)
Validation features: (17400, 1280)
Validation labels: (17400,)


In [10]:
#building and compiling our classifier model which will use precomputed MobileNetV2 features

feature_inputs = keras.Input(
    shape=(1280,)
)

x = keras.layers.Dense(
    128,
    activation="relu"
)(feature_inputs)

x = keras.layers.Dropout(0.3)(x)

feature_outputs = keras.layers.Dense(
    29,
    activation="softmax"
)(x)

classifier = keras.Model(
    feature_inputs,
    feature_outputs,
    name="signframe_classifier"
)


classifier.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

classifier.summary()

Model: "signframe_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 29)                  │           3,741 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 167,709 (655.11 KB)

 Trainable params: 167,709 (655.11 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:

feature_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),

    keras.callbacks.ModelCheckpoint(
        "best_signframe_feature_classifier.keras",
        monitor="val_loss",
        save_best_only=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=1,
        min_lr=1e-6
    )
]

history_features = classifier.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=feature_callbacks
)

Epoch 1/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.7012 - loss: 1.1610 - val_accuracy: 0.9489 - val_loss: 0.2908 - learning_rate: 1.0000e-04
Epoch 2/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.9263 - loss: 0.2999 - val_accuracy: 0.9790 - val_loss: 0.1277 - learning_rate: 1.0000e-04
Epoch 3/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9586 - loss: 0.1699 - val_accuracy: 0.9853 - val_loss: 0.0769 - learning_rate: 1.0000e-04
Epoch 4/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9728 - loss: 0.1128 - val_accuracy: 0.9922 - val_loss: 0.0503 - learning_rate: 1.0000e-04
Epoch 5/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9813 - loss: 0.0805 - val_accuracy: 0.9932 - val_loss: 0.0374 - learning_rate: 1.0000e-04
Epoch 6/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - accuracy: 0.9852 - loss: 0.0619 - val_accuracy: 0.9949 - val_loss: 0.0296 - learning_rate: 1.0000e-04
Epoch 7/10
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 17s 